# Figure 2 — Q1: do the evaluation criteria agree with each other?

If the ranking flips with the choice of distance or metric, every later result has to be read through that. This figure establishes how much it flips.

Each panel has its own cell and renders on its own, sized to suit it alone. The last cell assembles all of them at the submission size (180 x 225 mm) and writes the PDF.

The panel functions live in `figures/fig2.R` and are **sourced, not copied** — edit them there and re-run the cell, so the notebook and `snakemake` produce the same figure.

## Setup

In [ ]:
# Panels come from the script, so the notebook and the submitted figure cannot drift apart.
# main() is guarded, so sourcing runs nothing.
if (basename(getwd()) == "figures") setwd("..")
stopifnot("run this from benchmark/ or benchmark/figures/" = file.exists("config/methods.tsv"))
source("figures/fig2.R")

results_dir <- "results"

## Data

In [ ]:
ordination_set <- "real/hivrc_16s"   # 17 studies, the strongest batch structure of the evaluation sets
scored <- read_scores(file.path(results_dir, "scores.tsv"))
evaluated <- scored[scored$regime == "real" & scored$status == "ok", ]
shown <- intersect(c("raw", "limma", "conqur", "mmuphin", "harmony"),
                   evaluated$method[evaluated$dataset == ordination_set])

cat(nrow(evaluated), "scored method x dataset rows;  ordination shows:", paste(shown, collapse = ", "), "\n")

## a. Ordination before and after

One dataset, five methods, coloured by batch (top) and by phenotype (bottom). Points are drawn shuffled so the smaller phenotype group is not buried under the larger.

In [ ]:
show_panel(panel_ordination(results_dir, ordination_set, shown), 180, 58)

## b. Every method against every metric

Mean scaled score over the evaluation datasets. Grey means the metric does not apply to that output kind.

In [ ]:
show_panel(panel_global(evaluated), 98, 120)

## c. Does the ranking survive the distance?

Mean rank under three distances. Only the eight methods scorable under all three appear — Jaccard needs presence/absence.

In [ ]:
show_panel(panel_flip(evaluated), 98, 72)

## d. Agreement between metrics

Spearman correlation across all method x dataset results.

In [ ]:
show_panel(panel_metric_agreement(evaluated), 98, 85)

## e. The trade-off

Batch removal against signal retention, with the undominated methods joined by the Pareto front.

In [ ]:
show_panel(panel_tradeoff(evaluated), 170, 88)

## Assembled figure

Everything above, in the layout that goes to the journal: 180 x 225 mm, one page.
The panels shrink here — that is the real submission size, and the font sizes above were
chosen for it.

In [ ]:
figure <- (panel_ordination(results_dir, ordination_set, shown) /
           (panel_global(evaluated) | (panel_flip(evaluated) / panel_metric_agreement(evaluated))) /
           panel_tradeoff(evaluated)) +
  plot_layout(heights = c(0.9, 2.6, 1.3)) +
  plot_annotation(tag_levels = "a") &
  theme(plot.tag = element_text(size = 9, face = "bold"))

save_figure(figure, "results/figures/fig2.pdf", 180, 225)
show_panel(figure, 180, 225, scale = 1.5)